# IMEN266 · Ch.1–3 · Class Problems 2 — Continuous random variables & expectation

Same 5-step grammar as CP1: **Problem → Model → Analytic → Simulate → Experiment**.
The recurring craft in this notebook: *turn a pdf into (i) a cdf, (ii) a
probability, (iii) samples, (iv) moments* — and make all four agree.

▶ Colab: `https://colab.research.google.com/github/youngmko/imen266-2026/blob/main/ch01-03/notebooks/CP2_continuous_expectation.ipynb`

In [ ]:
# --- IMEN266 setup (self-contained: runs on Colab with zero installs) ---
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11,
                     "legend.frameon": False})

def check(name, got, expected, tol=None, rel=0.05):
    """Green/red self-check. Unfinished TODOs (None / ...) do not crash."""
    if got is None or got is Ellipsis:
        print(f"[TODO] {name}: not computed yet - fill the TODO above."); return False
    g = float(got)
    ok = (abs(g-expected) <= (rel*max(abs(expected),1e-12) if tol is None else tol))
    print(f"[{'OK ' if ok else 'X  '}] {name}: got {g:.6g}   (reference {expected:.6g})")
    return ok

try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider
    HAS_W = True
except Exception:
    HAS_W = False

def show(fn, **sliders):
    """interact() if widgets are available; otherwise draw once with defaults."""
    if HAS_W:
        interact(fn, **sliders)
    else:
        fn(**{k: getattr(v, "value", v) for k, v in sliders.items()})

print("Setup OK.  Widgets available:", HAS_W)

In [ ]:
from scipy.integrate import quad   # numerical integration, used throughout

---
## Problem 1 — A polynomial pdf *(slide p. 26/35)*

> Compute $F(x)$, the cdf, if $X$ is a continuous r.v. with pdf
> $$f(x)=\begin{cases}\tfrac34\,(1-x^2) & -1<x<1\\ 0 & \text{otherwise.}\end{cases}$$
> Also obtain the probability $P(-0.5 < X < 0.75)$.

### Model
$F(x)=\int_{-1}^{x}\tfrac34(1-t^2)\,dt=\tfrac{3x-x^3+2}{4}$ for $-1<x<1$
(and $0$/$1$ outside). Then $P(a<X<b)=F(b)-F(a)$.

In [ ]:
def f(x):
    x = np.asarray(x, float)
    return np.where((x > -1) & (x < 1), 0.75 * (1 - x**2), 0.0)

def F(x):
    """cdf. TODO 1: integrate f from -1 to x by hand ->  (3x - x^3 + 2)/4
    inside (-1,1); force 0 below -1 and 1 above 1."""
    x = np.asarray(x, float)
    inside = np.full_like(x, np.nan)   # TODO: replace with (3*x - x**3 + 2)/4
    return np.clip(np.where(x <= -1, 0.0, np.where(x >= 1, 1.0, inside)), 0, 1)

p_int = float(F(0.75) - F(-0.5))       # NaN until the TODO is done
print("P(-0.5 < X < 0.75) =", p_int)

In [ ]:
check("integral of f", quad(f, -1, 1)[0], 1.0, tol=1e-8)   # pdf property
check("F(0)  ", None if np.asarray(F(0.0)).dtype == object else float(F(0.0)), 0.5, tol=1e-9)
check("P(-0.5,0.75)", p_int, 0.80078125, tol=1e-9)

In [ ]:
# --- Sample from f by rejection, then let the histogram vote -----------------
def sample_f(n):
    out = np.empty(0)
    while out.size < n:
        x = rng.uniform(-1, 1, 2*n); u = rng.uniform(0, 0.75, 2*n)
        out = np.concatenate([out, x[u < f(x)]])
    return out[:n]

X = sample_f(200_000)
print(f"simulated P(-0.5<X<0.75) = {np.mean((X > -0.5) & (X < 0.75)):.4f}")

xs = np.linspace(-1.2, 1.2, 400)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(X, bins=60, density=True, alpha=.5, label="samples")
ax[0].plot(xs, f(xs), "k", label="f(x)")
m = (xs > -0.5) & (xs < 0.75)
ax[0].fill_between(xs[m], f(xs[m]), alpha=.35, color="#c0392b",
                   label="P(-0.5<X<0.75)")
ax[0].legend(); ax[0].set_title("pdf, samples, and the event as an area")
ax[1].plot(xs, F(xs)); ax[1].plot(np.sort(X), np.linspace(0, 1, X.size),
                                  alpha=.6, label="empirical cdf")
ax[1].set_title("F(x): formula vs empirical cdf"); ax[1].legend()
plt.tight_layout(); plt.show()

**Think about it** — (i) Verify the three cdf properties (limits at
$\pm\infty$, monotone, right-continuity) directly on the plot. (ii) Probability
as *area under the pdf* vs as a *height difference on the cdf* — same number,
two pictures. Which is easier to read off for a tiny interval?

---
## Problem 2 — Exponential lifetime *(slide p. 29/35)*

> The lifetime of a single-cell organism is exponentially distributed with
> parameter 0.1 (per hour). What is the probability that this organism will
> live for more than 20 hours? What is the probability that it will die
> within 5 hours?

### Model
$X\sim\text{Exp}(\lambda=0.1)$: $\ P(X>t)=e^{-\lambda t}$, $\ P(X\le t)=1-e^{-\lambda t}$.

In [ ]:
lam = 0.1
# TODO 2: survival and cdf of the exponential
p_live20 = ...
p_die5   = ...
print("P(X > 20) =", p_live20, "   P(X <= 5) =", p_die5)

In [ ]:
check("P(X>20)", p_live20, math.exp(-2.0), tol=1e-12)
check("P(X<=5)", p_die5, 1 - math.exp(-0.5), tol=1e-12)

In [ ]:
X = rng.exponential(scale=1/0.1, size=300_000)          # mean 10 h
print(f"sim: P(X>20) = {np.mean(X > 20):.4f}   P(X<=5) = {np.mean(X <= 5):.4f}")

ts = np.linspace(0, 60, 300)
plt.figure()
plt.semilogy(ts, np.exp(-0.1 * ts), "k", label=r"$e^{-0.1t}$")
plt.semilogy(np.sort(X), 1 - np.linspace(0, 1, X.size, endpoint=False),
             alpha=.7, label="empirical survival")
plt.xlabel("t (hours)"); plt.ylabel("P(X > t)   (log scale)")
plt.title("Exponential survival is a straight line on a log scale")
plt.legend(); plt.show()

In [ ]:
# --- (Ch.5 preview, optional) memorylessness in one line ---------------------
lhs = np.mean(X[X > 5] > 25)      # P(X>25 | X>5)
rhs = np.mean(X > 20)             # P(X>20)
print(f"P(X>25 | X>5) = {lhs:.4f}   vs   P(X>20) = {rhs:.4f}   -> 'good as new'")

**Think about it** — the straight line on the semi-log plot *is* the
exponential's fingerprint (constant hazard). Keep this picture: in Ch.5 it
becomes the memoryless property, and in Ch.9 the "failure rate".

---
## Problem 3 — Erlang message delay *(slide p. 29/35)*

> The time for a message to reach www.microsoft.com from POSTECH is
> distributed according to an Erlang distribution with mean 300 ms and
> variance 30000 squared-ms. What is the probability that a message will
> reach within 600 ms? What is the probability it will take more than 900 ms?

### Model
Erlang$(k,\lambda)$: mean $=k/\lambda$, variance $=k/\lambda^2$, hence
$$k=\frac{\text{mean}^2}{\text{var}},\qquad \lambda=\frac{k}{\text{mean}},\qquad
P(T\le t)=1-e^{-\lambda t}\sum_{n=0}^{k-1}\frac{(\lambda t)^n}{n!}.$$

In [ ]:
mean, var = 300.0, 30_000.0

# TODO 3a: recover the Erlang parameters from mean & variance
k   = ...        # shape  (integer!)  hint: mean^2 / var
lam = ...        # rate               hint: k / mean

def erlang_cdf(t, k, lam):
    return 1 - math.exp(-lam*t) * sum((lam*t)**n / math.factorial(n)
                                      for n in range(k))

p_600  = erlang_cdf(600, k, lam) if k is not Ellipsis else None
p_900p = 1 - erlang_cdf(900, k, lam) if k is not Ellipsis else None
print("k, lam =", k, lam); print("P(T<=600) =", p_600, "  P(T>900) =", p_900p)

In [ ]:
check("k",        k,   3,    tol=1e-12)
check("lambda",   lam, 0.01, tol=1e-12)
check("P(T<=600)", p_600,  1 - 25*math.exp(-6), tol=1e-9)
check("P(T>900)",  p_900p, 50.5*math.exp(-9),   tol=1e-9)

In [ ]:
# --- The Erlang story: a sum of k exponential phases -------------------------
T = rng.exponential(scale=100.0, size=(300_000, 3)).sum(axis=1)  # 3 x Exp(0.01/ms)
print(f"sim: P(T<=600) = {np.mean(T <= 600):.4f}    P(T>900) = {np.mean(T > 900):.5f}")

ts = np.linspace(0, 1200, 400)
plt.figure()
plt.hist(T, bins=80, density=True, alpha=.5, label="sum of 3 Exp(0.01)")
plt.plot(ts, stats.gamma.pdf(ts, a=3, scale=100), "k", label="Erlang(3, 0.01) pdf")
plt.fill_between(ts[ts > 900], stats.gamma.pdf(ts[ts > 900], a=3, scale=100),
                 color="#c0392b", alpha=.4, label="P(T > 900)")
plt.xlabel("delay (ms)"); plt.legend()
plt.title("Erlang = k exponential phases in series (Ch.5: phase-type thinking)")
plt.show()

**Think about it** — (i) Same mean 300 ms but $k=1$ (pure exponential):
does $P(T>900)$ go up or down? Check with the cdf. (ii) Matching *two*
moments pinned down *two* parameters — this "moment matching" trick returns in
Ch.8 for M/G/1 queues.

---
## Problem 4 — Find $a$ and $b$ *(slide p. 34/35)*

> If the cdf of a continuous r.v. is given by
> $$F(x)=\begin{cases}0 & x<0\\ ax+\dfrac{bx^3}{3} & 0\le x\le 1\\ 1 & x>1\end{cases}$$
> and $E[X]=0.6$, **find $a$ and $b$.**

### Model — two conditions, two unknowns
1. $F(1)=1\ \Rightarrow\ a+\dfrac b3=1$
2. $f(x)=F'(x)=a+bx^2$, so $E[X]=\displaystyle\int_0^1 x\,(a+bx^2)\,dx=\dfrac a2+\dfrac b4=0.6$

In [ ]:
# TODO 4: write the 2x2 linear system  A @ [a, b] = rhs  and solve it.
#   row 1:  F(1)=1      ->  a*1   + b*(1/3) = 1
#   row 2:  E[X]=0.6    ->  a*(1/2) + b*(1/4) = 0.6
A   = ...
rhs = ...
a, b = (np.linalg.solve(A, rhs) if A is not Ellipsis else (None, None))
print("a =", a, " b =", b)

In [ ]:
check("a", a, 0.6, tol=1e-9)
check("b", b, 1.2, tol=1e-9)

In [ ]:
# --- Trust, then verify: is (a,b) really a valid distribution with E=0.6? ---
if a is None:
    print("finish TODO 4 first, then rerun this cell")
else:
    f_ = lambda x: a + b*x**2
    print("f >= 0 on [0,1]? ", f_(0) >= 0 and f_(1) >= 0)        # convex in x^2
    print(f"integral f  = {quad(f_, 0, 1)[0]:.6f}   (should be 1)")
    print(f"E[X]        = {quad(lambda x: x*f_(x), 0, 1)[0]:.6f}   (should be 0.6)")

    xs = np.linspace(0, 1, 200)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(xs, f_(xs)); ax[0].set_title("pdf  f(x) = a + b x²")
    ax[1].plot(xs, a*xs + b*xs**3/3); ax[1].set_title("cdf  F(x)")
    for A_ in ax: A_.set_xlabel("x")
    plt.tight_layout(); plt.show()

**Think about it** — we used exactly two facts: *total probability is 1* and
*the stated mean*. If instead of $E[X]$ you were given $P(X\le 0.5)=0.4$,
set up (don't solve) the new system.

---
## Problem 5 — Document sizes (heavy tail!) *(slide p. 35/35)*

> If $X$ is a continuous r.v. denoting the size of a document at a web server
> with pdf
> $$f(x)=\begin{cases}\dfrac{24}{x^4} & x\ge 2\\ 0 & \text{otherwise},\end{cases}$$
> **compute the variance of $X$.**

### Model
$E[X]=\displaystyle\int_2^\infty \frac{24}{x^3}dx,\qquad
E[X^2]=\displaystyle\int_2^\infty \frac{24}{x^2}dx,\qquad
\mathrm{Var}=E[X^2]-E[X]^2 .$

In [ ]:
# TODO 5: two integrals by hand, then the variance identity.
#   ∫_2^∞ 24 x^-3 dx = ?        ∫_2^∞ 24 x^-2 dx = ?
EX  = ...
EX2 = ...
Var = EX2 - EX**2 if EX is not Ellipsis else None
print("E[X] =", EX, " E[X^2] =", EX2, " Var =", Var)

In [ ]:
check("E[X]",  EX,  3.0,  tol=1e-9)
check("E[X^2]", EX2, 12.0, tol=1e-9)
check("Var",   Var, 3.0,  tol=1e-9)

In [ ]:
# --- Sample by inverse transform:  F(x)=1-8/x^3  ->  X = 2 (1-U)^{-1/3} ------
U = rng.random(300_000)
X = 2 * (1 - U) ** (-1/3)
print(f"sample mean {X.mean():.3f} (→3)    sample var {X.var():.3f} (→3, slowly!)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
xs = np.linspace(2, 20, 300)
ax[0].hist(X[X < 20], bins=80, density=True, alpha=.5, label="samples")
ax[0].plot(xs, 24/xs**4, "k", label=r"$24/x^4$"); ax[0].set_yscale("log")
ax[0].set_title("heavy tail (log y)"); ax[0].legend()
n = np.arange(1, X.size + 1)
ax[1].plot(n, X.cumsum()/n); ax[1].axhline(3, color="k", ls="--")
ax[1].set_xscale("log"); ax[1].set_title("running mean → 3, but bumpy: rare huge files")
ax[1].set_xlabel("n samples")
plt.tight_layout(); plt.show()

**Think about it** — (i) For $f(x)=c/x^4$, which moments are finite? What if
the tail were $c/x^3$? (ii) Web-file sizes really are heavy-tailed — in Ch.8
this is exactly why M/G/1 waiting times blow up with service *variability*,
not just service *mean*.

---
## Bridge → Companion Proof 3 (tail-sum formula)

For $X\ge 0$: $\;E[X]=\displaystyle\int_0^\infty P(X>x)\,dx$.
Before reading the proof, watch it be true:

In [ ]:
# Problem 2:  Exp(0.1) has E[X]=10;   Problem 5:  Pareto has E[X]=3
for name, S, ref in [("Exp(0.1)",  lambda x: np.exp(-0.1*x), 10.0),
                     ("24/x^4 doc", lambda x: np.where(x < 2, 1.0, 8.0/x**3), 3.0)]:
    val = quad(S, 0, np.inf)[0]
    print(f"{name:12s}  ∫P(X>x)dx = {val:.4f}   E[X] = {ref}")

Both match. *Now* go read why (Companion, Proof 3 — do the gap-fill version
first), then answer its Checkpoint questions.

---
## Wrap-up

| # | pdf/cdf | skill practiced | answer |
|---|---|---|---|
| 1 | $\tfrac34(1-x^2)$ | pdf→cdf→prob, rejection sampling | $P=0.8008$ |
| 2 | Exp(0.1) | survival function, log-scale eye | $e^{-2}\!=\!.1353$ |
| 3 | Erlang | moment matching, phase story | $k{=}3,\ \lambda{=}.01$ |
| 4 | $ax+bx^3/3$ | constraints → parameters | $a{=}.6,\ b{=}1.2$ |
| 5 | $24/x^4$ | LOTUS, variance, heavy tails | $\mathrm{Var}=3$ |

**Next** → Companion Proofs 3–5 → HW 1.